# MGMT298D: Science and Strategy of AI
## Week 4 Assignment - Reinforcement Learning
### Application: E-commerce Price Optimization

---

**Instructions:** Complete the exercises below by filling in the `???` placeholders and answering the questions. Run all code cells in order.

**Key Concept:** In reinforcement learning, an agent learns by trial and error. The core challenge is **exploration vs. exploitation**: try new things or stick with what works?

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

# THE ENVIRONMENT (agent doesn't know these true rates!)
PRICES = [5, 10, 15, 20, 25]
TRUE_CONVERSION_RATES = [0.50, 0.35, 0.22, 0.12, 0.05]

def simulate_sale(price_index):
    """Simulate one customer. Returns revenue (price if buy, else 0)."""
    bought = np.random.random() < TRUE_CONVERSION_RATES[price_index]
    return PRICES[price_index] if bought else 0

# Show true expected revenue (agent must discover this!)
expected_revenues = [p * r for p, r in zip(PRICES, TRUE_CONVERSION_RATES)]

plt.figure(figsize=(8, 4))
bars = plt.bar(['$5', '$10', '$15', '$20', '$25'], expected_revenues, color='steelblue', edgecolor='black')
plt.ylabel('Expected Revenue per Customer ($)')
plt.title('True Expected Revenue by Price\n(Agent must DISCOVER this through trial and error!)')
for bar, rev in zip(bars, expected_revenues):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, f'${rev:.2f}', ha='center', fontweight='bold')
plt.ylim(0, 4)
plt.show()

print(f"Optimal price: ${PRICES[expected_revenues.index(max(expected_revenues))]}")

---

## Part 1: Multi-Armed Bandits

Three strategies:
- **Random**: Pick randomly (baseline)
- **ε-Greedy**: Usually pick best, but explore randomly ε% of time
- **UCB**: Pick based on average + uncertainty bonus

In [ ]:
def run_bandit(strategy, epsilon, n_customers, ucb_confidence=2.0):
    """Run a bandit algorithm."""
    counts = [0]*5
    total_rewards = [0]*5
    history = []
    cumulative = 0
    
    for t in range(n_customers):
        if strategy == 'random':
            choice = np.random.randint(5)
        elif strategy == 'epsilon-greedy':
            if min(counts) == 0:
                choice = counts.index(0)
            elif np.random.random() < epsilon:
                choice = np.random.randint(5)
            else:
                avgs = [total_rewards[i]/counts[i] for i in range(5)]
                choice = avgs.index(max(avgs))
        elif strategy == 'ucb':
            if min(counts) == 0:
                choice = counts.index(0)
            else:
                ucb_scores = []
                for i in range(5):
                    avg = total_rewards[i] / counts[i]
                    bonus = ucb_confidence * np.sqrt(np.log(t + 1) / counts[i])
                    ucb_scores.append(avg + bonus)
                choice = ucb_scores.index(max(ucb_scores))
        
        revenue = simulate_sale(choice)
        counts[choice] += 1
        total_rewards[choice] += revenue
        cumulative += revenue
        history.append(cumulative)
    
    return history, counts, total_rewards

In [ ]:
# ============================================================
# EXERCISE 1: Tune the exploration parameter (epsilon)
# ============================================================
# epsilon controls how often the agent explores randomly
# Try: 0 (never explore), 0.05, 0.1, 0.2, 0.5 (explore half the time)

EPSILON = ???  # <-- Fill in a value between 0 and 1
UCB_CONFIDENCE = 2.0
N_CUSTOMERS = 1000

# Run strategies
random_hist, random_counts, _ = run_bandit('random', 0, N_CUSTOMERS)
eg_hist, eg_counts, _ = run_bandit('epsilon-greedy', EPSILON, N_CUSTOMERS)
ucb_hist, ucb_counts, _ = run_bandit('ucb', 0, N_CUSTOMERS, UCB_CONFIDENCE)

# Plot results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Revenue over time
axes[0].plot(random_hist, label='Random', alpha=0.7)
axes[0].plot(eg_hist, label=f'ε-Greedy (ε={EPSILON})', linewidth=2)
axes[0].plot(ucb_hist, label=f'UCB (c={UCB_CONFIDENCE})', linewidth=2)
axes[0].set_xlabel('Customer #', fontsize=12)
axes[0].set_ylabel('Cumulative Revenue ($)', fontsize=12)
axes[0].set_title('Which Strategy Earns the Most?', fontsize=14)
axes[0].legend()

# Price selection frequency
x = np.arange(5)
width = 0.25
axes[1].bar(x - width, random_counts, width, label='Random', alpha=0.7)
axes[1].bar(x, eg_counts, width, label='ε-Greedy', alpha=0.7)
axes[1].bar(x + width, ucb_counts, width, label='UCB', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(['$5', '$10', '$15', '$20', '$25'])
axes[1].set_xlabel('Price', fontsize=12)
axes[1].set_ylabel('Times Chosen', fontsize=12)
axes[1].set_title('Which Prices Did Each Strategy Try?', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Final Revenue after {N_CUSTOMERS} customers:")
print(f"  Random:    ${random_hist[-1]:,}")
print(f"  ε-Greedy:  ${eg_hist[-1]:,}")
print(f"  UCB:       ${ucb_hist[-1]:,}")

**Q1:** Which strategy earned the most revenue? Looking at the bar chart, did the best strategy correctly identify the optimal price ($10)?

*Your answer:*


In [ ]:
# ============================================================
# EXERCISE 2: Explore the exploration-exploitation trade-off
# ============================================================
# Test different epsilon values

epsilons = [0, 0.01, 0.05, 0.1, 0.2, 0.3, 0.5]
final_revenues = []

for eps in epsilons:
    hist, _, _ = run_bandit('epsilon-greedy', eps, 1000)
    final_revenues.append(hist[-1])

plt.figure(figsize=(10, 5))
plt.plot(epsilons, final_revenues, 'bo-', linewidth=2, markersize=10)
plt.xlabel('Epsilon (Exploration Rate)', fontsize=12)
plt.ylabel('Total Revenue after 1000 Customers ($)', fontsize=12)
plt.title('Effect of Exploration Rate on Revenue\n(0 = Pure Exploitation, 1 = Pure Exploration)', fontsize=14)
plt.grid(alpha=0.3)

# Mark best epsilon
best_idx = final_revenues.index(max(final_revenues))
plt.scatter([epsilons[best_idx]], [final_revenues[best_idx]], color='red', s=200, zorder=5, label=f'Best: ε={epsilons[best_idx]}')
plt.legend()
plt.show()

**Q2:** What happens when epsilon = 0 (no exploration)? What happens when epsilon is very high (too much exploration)? What's the "sweet spot"?

*Your answer:*


---

## Part 2: Q-Learning

Q-Learning adds **state awareness**: the best price might depend on the situation (inventory level, time in season).

In [ ]:
class PricingGame:
    """Inventory-based pricing game."""
    def __init__(self, inventory=20, time_limit=20):
        self.start_inventory = inventory
        self.time_limit = time_limit
        self.reset()
    
    def reset(self):
        self.inventory = self.start_inventory
        self.time_left = self.time_limit
        return self.get_state()
    
    def get_state(self):
        inv = 'high' if self.inventory > 10 else 'low'
        time = 'early' if self.time_left > 10 else 'late'
        return (inv, time)
    
    def step(self, price_index):
        if self.time_left <= 0 or self.inventory <= 0:
            return self.get_state(), 0, True
        
        # Urgency increases conversion near deadline
        base_rate = TRUE_CONVERSION_RATES[price_index]
        urgency = 1 + 0.5 * (1 - self.time_left / self.time_limit)
        rate = min(0.9, base_rate * urgency)
        
        sold = np.random.random() < rate
        revenue = PRICES[price_index] if sold else 0
        
        if sold:
            self.inventory -= 1
        self.time_left -= 1
        
        done = self.time_left <= 0 or self.inventory <= 0
        return self.get_state(), revenue, done

def run_qlearning(learning_rate, discount, epsilon, n_episodes):
    """Train a Q-learning agent."""
    Q = {
        ('high', 'early'): [0]*5,
        ('high', 'late'):  [0]*5,
        ('low', 'early'):  [0]*5,
        ('low', 'late'):   [0]*5,
    }
    
    game = PricingGame()
    episode_revenues = []
    
    for episode in range(n_episodes):
        state = game.reset()
        total_revenue = 0
        
        while True:
            # Choose action (epsilon-greedy)
            if np.random.random() < epsilon:
                action = np.random.randint(5)
            else:
                action = Q[state].index(max(Q[state]))
            
            next_state, revenue, done = game.step(action)
            total_revenue += revenue
            
            # Q-learning update
            old_value = Q[state][action]
            if done:
                target = revenue
            else:
                target = revenue + discount * max(Q[next_state])
            Q[state][action] = old_value + learning_rate * (target - old_value)
            
            state = next_state
            if done:
                break
        
        episode_revenues.append(total_revenue)
    
    return episode_revenues, Q

In [ ]:
# ============================================================
# EXERCISE 3: Tune Q-Learning parameters
# ============================================================

LEARNING_RATE = ???  # How fast agent updates beliefs. Try: 0.01, 0.1, 0.5
DISCOUNT = ???       # How much agent values future vs now. Try: 0, 0.5, 0.9, 0.99
EPSILON = 0.2        # Exploration rate
N_EPISODES = 500

# Train the agent
revenues, Q_table = run_qlearning(LEARNING_RATE, DISCOUNT, EPSILON, N_EPISODES)

# Plot learning curve
plt.figure(figsize=(10, 5))
window = 20
smoothed = [np.mean(revenues[max(0,i-window):i+1]) for i in range(len(revenues))]

plt.plot(revenues, alpha=0.3, color='blue')
plt.plot(smoothed, color='blue', linewidth=2, label='Smoothed Average')
plt.xlabel('Episode', fontsize=12)
plt.ylabel('Total Revenue ($)', fontsize=12)
plt.title(f'Q-Learning: Revenue Improves Over Time\n(LR={LEARNING_RATE}, Discount={DISCOUNT}, ε={EPSILON})', fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Average revenue (first 50 episodes):  ${np.mean(revenues[:50]):.1f}")
print(f"Average revenue (last 50 episodes):   ${np.mean(revenues[-50:]):.1f}")

**Q3:** Did the agent improve over time (compare first 50 vs last 50 episodes)? What does the discount factor control—why might a high discount (0.9+) be important for this pricing problem?

*Your answer:*


In [ ]:
# Visualize the learned policy
fig, ax = plt.subplots(figsize=(8, 5))

policy_grid = []
for inv in ['high', 'low']:
    row = []
    for time in ['early', 'late']:
        best_action = Q_table[(inv, time)].index(max(Q_table[(inv, time)]))
        row.append(PRICES[best_action])
    policy_grid.append(row)

im = ax.imshow(policy_grid, cmap='RdYlGn_r', vmin=5, vmax=25)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Early in Season', 'Late in Season'])
ax.set_yticks([0, 1])
ax.set_yticklabels(['High Inventory', 'Low Inventory'])

for i in range(2):
    for j in range(2):
        ax.text(j, i, f'${policy_grid[i][j]}', ha='center', va='center', fontsize=24, fontweight='bold')

plt.title('Learned Pricing Policy: Best Price by Situation', fontsize=14)
plt.colorbar(im, label='Price ($)')
plt.tight_layout()
plt.show()

**Q4:** Look at the learned policy grid. Does the pricing strategy make business sense? For example:
- What price does it recommend when inventory is HIGH and it's LATE in the season?
- What about when inventory is LOW and it's EARLY?
- Why might these recommendations make sense?

*Your answer:*


In [ ]:
# ============================================================
# EXERCISE 4: Compare discount factors
# ============================================================

discounts = [0, 0.5, 0.9, 0.99]
results = {}

for d in discounts:
    revs, Q = run_qlearning(0.1, d, 0.2, 500)
    results[d] = {'revenues': revs, 'Q': Q}

# Plot comparison
plt.figure(figsize=(10, 5))
for d in discounts:
    smoothed = [np.mean(results[d]['revenues'][max(0,i-20):i+1]) for i in range(500)]
    plt.plot(smoothed, label=f'Discount = {d}', linewidth=2)

plt.xlabel('Episode', fontsize=12)
plt.ylabel('Revenue (smoothed)', fontsize=12)
plt.title('Effect of Discount Factor on Learning', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

---

## Question 5: Business Application

**Q5a:** An e-commerce manager says: "Why should I let an algorithm randomly try prices? That could lose us money!" How would you explain the value of exploration to this skeptic?

*Your answer:*


**Q5b:** In what types of business situations would you recommend using reinforcement learning for pricing vs. a simple A/B test? What are the trade-offs?

*Your answer:*


---

## Question 6: Ethical Considerations

**Q6:** Dynamic pricing algorithms (like Uber surge pricing) have faced criticism for being unfair to consumers. What ethical concerns might arise from using reinforcement learning for pricing? How might a company address these concerns while still optimizing revenue?

*Your answer:*
